# SHEESHA - one-shot test clip (no Google Drive needed)

Generates the test clip from episode 1 and downloads it straight to
your machine. This replaces the Drive-based notebook, which intermittently
fails with `credential propagation was unsuccessful`.

**Do these three things, in order:**

1. **Runtime > Change runtime type > T4 GPU**, then Save. Skip this and the
   GPU is not attached.
2. Run every cell below, top to bottom. The model cell downloads about 29 GB
   the first time; the last cell downloads your clip to the normal Downloads
   folder.
3. When the clip lands, copy it into this project at `render\clips\wan\` and
   run `python pipeline.py preview 1`.

Expected file: `ep01_s6_doorway-figure_ai.mp4`

Rehearsal settings: 49 frames at 20 steps, not the full
81/30. That is deliberate: the ~29 GB model download is the unavoidable cost
(about 17 minutes, and this repo ships fp32 weights only), so generation time
is what gets sacrificed. It is still enough to judge the look and the motion.

Budget roughly 29 minutes total, and run it straight through in one
go. An interrupted session loses the download and has to start again. Keep the
tab focused, do not let the laptop sleep, and do not click Stop. If it does
disconnect, re-run the last two cells; finished work is skipped.

In [ ]:
!pip install -q --upgrade "diffusers>=0.33" transformers accelerate imageio imageio-ffmpeg ftfy sentencepiece

In [ ]:
# The job list travels inside this notebook, so nothing has to be uploaded
# and Drive is never touched.
import base64, json, pathlib, time
import torch
MANIFEST_B64 = "eyJlcGlzb2RlIjogMSwgInNjaGVtYSI6ICJzaGVlc2hhLnZpZGVvX2pvYnMvMSIsICJtb2RlbCI6ICJXYW4tQUkvV2FuMi4xLVQyVi0xLjNCLURpZmZ1c2VycyIsICJzdGVwcyI6IDIwLCAiZ3VpZGFuY2Vfc2NhbGUiOiA2LjAsICJkZWxpdmVyeSI6IHsid2lkdGgiOiAxMDgwLCAiaGVpZ2h0IjogMTkyMCwgImZwcyI6IDI0LCAiZHVyYXRpb25fcyI6IDMwLjB9LCAibm90ZXMiOiBbIkdlbmVyYXRlIGV2ZXJ5IGpvYiBpbnRvIG91dF9uYW1lIGluc2lkZSB0aGUgb3V0cHV0IGRpcmVjdG9yeS4iLCAibnVtX2ZyYW1lcyBtdXN0IGJlIDRrKzE7IDgxIGlzIDgxIGZyYW1lcyBhdCAxNiBmcHMuIiwgImZsb2F0MTYgaXMgcmVxdWlyZWQ6IGEgQ29sYWIgVDQgaXMgVHVyaW5nIChzbV83NSkgYW5kIGhhcyBubyBiZmxvYXQxNi4iLCAiVEVTVCBNQU5JRkVTVDogb25lIHNob3Qgb25seSwgdG8gdmFsaWRhdGUgdGhlIGxvb2sgYmVmb3JlIGNvbW1pdHRpbmcgdG8gYSBmdWxsIGVwaXNvZGUuIl0sICJqb2JzIjogW3siaWQiOiAiczYiLCAia2luZCI6ICJkb29yd2F5X2ZpZ3VyZSIsICJvdXRfbmFtZSI6ICJlcDAxX3M2X2Rvb3J3YXktZmlndXJlX2FpLm1wNCIsICJwcm9tcHQiOiAiY2luZW1hdGljIGhvcnJvciBzY2VuZSwgY29udGludW91cyBzaW5nbGUgdGFrZSwgcGhvdG9yZWFsaXN0aWMsIGxpdmUgYWN0aW9uIGZpbG0gc3RpbGwgYWVzdGhldGljLCBzaG90IG9uIDM1bW0gYW5hbW9ycGhpYyBsZW5zLCBzaGFsbG93IGRlcHRoIG9mIGZpZWxkLCBtb29keSBsb3cta2V5IGxpZ2h0aW5nLCBkZWVwIHNoYWRvd3MsIG11dGVkIGRlc2F0dXJhdGVkIHBhbGV0dGUsIHZvbHVtZXRyaWMgaGF6ZSwgc3VidGxlIGZpbG0gZ3JhaW4sIGhhbmRoZWxkIG1pY3JvIGRyaWZ0LCBzbG93IGRlbGliZXJhdGUgY2FtZXJhIG1vdmVtZW50LCBoaWdoIGNvbnRyYXN0IGZhbGxvZmYsIGZpcnN0IHBlcnNvbiBQT1YsIG5vIGRheWxpZ2h0LCBuaWdodCBpbnRlcmlvciBvZiBhbiBvbGQgTm9ydGggSW5kaWFuIGhhdmVsaSwgcHJhY3RpY2FsIGxhbXAgbGlnaHQgb25seSwgZmlyc3QgcGVyc29uIFBPViBvZiBhIGRhcmsgZG9vcndheSBiYWNrbGl0IGZyb20gYmVoaW5kLCBhIHRhbGwgZmFjZWxlc3MgZmVtYWxlIHNpbGhvdWV0dGUgc3RhbmRpbmcgaW4gdGhlIGZyYW1lIHdpdGggYSBkcmFwZWQgY2xvdGgsIGZvZywgY2luZW1hdGljIGhvcnJvciwgc3RhdGljLCB1bmlkZW50aWZpYWJsZSBmaWd1cmUsIGZhY2UgaGlkZGVuLCBiYWNrbGl0IHNpbGhvdWV0dGUgb25seSIsICJuZWdhdGl2ZV9wcm9tcHQiOiAid29yc3QgcXVhbGl0eSwgbG93IHF1YWxpdHksIEpQRUcgY29tcHJlc3Npb24gcmVzaWR1ZSwgYmx1cnJ5LCBvdmVyZXhwb3NlZCwgYnJpZ2h0IHRvbmVzLCBzdWJ0aXRsZXMsIHRleHQgb3ZlcmxheSwgd2F0ZXJtYXJrLCBsb2dvLCBwYWludGluZywgc3RpbGwgcGljdHVyZSwgc3RhdGljIGltYWdlLCBmcm96ZW4gZnJhbWUsIGNhcnRvb24sIGFuaW1lLCAzZCByZW5kZXIsIGNnaSwgZGVmb3JtZWQsIGRpc2ZpZ3VyZWQsIG1pc3NoYXBlbiBsaW1icywgZXh0cmEgZmluZ2VycywgZnVzZWQgZmluZ2VycywgcG9vcmx5IGRyYXduIGhhbmRzLCBwb29ybHkgZHJhd24gZmFjZXMsIHZpc2libGUgZmFjZSwgZGV0YWlsZWQgZmFjZSwgcG9ydHJhaXQsIGNyb3dkZWQgYmFja2dyb3VuZCwgbWFueSBwZW9wbGUsIGNhbWVyYSBzaGFrZSwgbW9ycGhpbmcsIGZsaWNrZXJpbmcsIGZhY2lhbCBmZWF0dXJlcywgZXllcyIsICJzZWVkIjogMjA0NTgxMDYsICJoZWlnaHQiOiA4MzIsICJ3aWR0aCI6IDQ4MCwgIm51bV9mcmFtZXMiOiA0OSwgImZwcyI6IDE2LCAid2FudF9mcmFtZXMiOiAxMDMsICJ3YW50X2R1cmF0aW9uIjogNC4yOTE3LCAibmFycmF0aW9uIjogIk1lcmEgY2hlaHJhIG5haGkgdGhhLCB1c2tlIHBlZWNoaGUga29pIG5haGkgdGhhLiJ9XX0="
manifest = json.loads(base64.b64decode(MANIFEST_B64))
jobs = manifest["jobs"]
OUT = pathlib.Path("out"); OUT.mkdir(exist_ok=True)

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"
vram = (torch.cuda.get_device_properties(0).total_memory / 2**30
        if torch.cuda.is_available() else 0)
print(f"GPU     : {gpu} ({vram:.1f} GB)")
print(f"episode : {manifest['episode']}   jobs: {len(jobs)}")
for j in jobs:
    print(f"  {j['id']} {j['kind']} {j['width']}x{j['height']} "
          f"{j['num_frames']}f seed={j['seed']}")
print(f"bfloat16 available: {torch.cuda.is_bf16_supported()}")
if "T4" not in gpu:
    print("\nWARNING: no T4. A GPU with under 10 GB will probably OOM.")


In [ ]:
# float16, never bfloat16: a T4 is Turing (sm_75) and has no bf16 support.
from diffusers import AutoencoderKLWan, WanPipeline

MODEL = manifest["model"]
print(f"loading {MODEL} ... (~9 GB of weights, this takes a few minutes)")
vae = AutoencoderKLWan.from_pretrained(
    MODEL, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL, vae=vae, torch_dtype=torch.float16)
pipe.set_progress_bar_config(disable=True)
OFFLOAD = False   # set True if you get CUDA out of memory
pipe.enable_model_cpu_offload() if OFFLOAD else pipe.to("cuda")
print("model ready")


In [ ]:
from diffusers.utils import export_to_video

for n, job in enumerate(jobs, 1):
    out = OUT / job["out_name"]
    if out.exists() and out.stat().st_size > 20_000:
        print(f"[{n}/{len(jobs)}] {job['out_name']} already done, skipping")
        continue
    print(f"[{n}/{len(jobs)}] {job['id']} {job['kind']} seed={job['seed']}")
    t0 = time.time()
    gen = torch.Generator(device="cuda").manual_seed(int(job["seed"]))
    frames = pipe(
        prompt=job["prompt"],
        negative_prompt=job["negative_prompt"],
        height=int(job["height"]), width=int(job["width"]),
        num_frames=int(job["num_frames"]),
        guidance_scale=float(manifest["guidance_scale"]),
        num_inference_steps=int(manifest["steps"]),
        generator=gen,
    ).frames[0]
    export_to_video(frames, str(out), fps=int(job["fps"]))
    torch.cuda.empty_cache()
    print(f"    done in {time.time() - t0:.0f}s -> {out.name}")
print("generation complete")


In [ ]:
# Hands the clip to your browser's normal download folder.
from google.colab import files

for job in jobs:
    p = OUT / job["out_name"]
    if p.exists() and p.stat().st_size > 20_000:
        print(f"downloading {p.name} ({p.stat().st_size/1e6:.1f} MB)")
        files.download(str(p))
    else:
        print(f"MISSING: {p.name}")
